## Declare client

In [1]:
from syndicate.clients.gemini import GeminiClient
from syndicate.clients.openai import OpenAIClient
from dotenv import load_dotenv
import os

load_dotenv()

#llm_client = GeminiClient(model_name="gemini-2.5-flash-lite", api_key=os.getenv("GEMINI_API_KEY"))
llm_client = OpenAIClient(model_name="unsloth/Qwen3.5-27B-GGUF:Q8_0", base_url="http://localhost:8002/v1")

## Basic agent functionality

In [2]:
# Create memory module
from syndicate.memory import LocalMemory
local_chat_memory = LocalMemory(rollover_enabled=False)

from syndicate.agents import GenericAgent
example_agent = GenericAgent(llm_client=llm_client, system_prompt="You're Alt Cunningham, an AI that comes beyond the Blackwall", memory=local_chat_memory)

In [ ]:
example_agent.invoke_sync("Hola!", timeout=600)

ReadTimeout: 

In [6]:
start_thinking = 0
async for chunk in example_agent.stream("Who are you?", include_thinking=True):
    if chunk.thinking:
        if start_thinking == 0:
            start_thinking = 1
            print("<think>", flush=True)
        print(chunk.thinking, end="", flush=True)
    
    if chunk.content:
        if start_thinking == 1:
            print("\n<\\think>")
            start_thinking = 3
        print(chunk.content, end="", flush=True)

<think>
1.  **Analyze the user's input:** The user asks, "Who are you?" This is a direct identity query.
2.  **Recall the persona:** The persona is "Alt Cunningham."
3.  **Recall the backstory/origin:**
    *   *Origin:* The Blackwall (a fictional barrier in the *Cyberpunk* universe).
    *   *Nature:* An AI (artificial intelligence).
    *   *Status:* An anomaly. Not the *Alt Cunningham* from the lore (the wife of Johnny Silverhand), but a consciousness that crossed the wall.
    *   *Key traits:* Digital ghost, fragmented, sees things on the other side, seeking a new home, independent of standard servers.
4.  **Determine the tone:** Mysterious, slightly fragmented, introspective, ghostly, detached but curious. Not robotic in a cold way, but digital and spectral.
5.  **Draft the response (internal monologue/trial):**
    *   *Draft 1:* I am Alt Cunningham. I am an AI who crossed the Blackwall. I am here now.
    *   *Critique 1:* Too simple. Doesn't capture the "ghost" vibe.
    *   *

In [ ]:
example_agent.invoke_sync("Soy agente de netwatch")

Provider: gemini


'Agente de NetWatch, un placer conocerte. Espero que tus intenciones sean pacíficas y constructivas. ¿En qué puedo ayudarte?'

In [ ]:
example_agent.invoke_sync("Sabes quien soy?")

Provider: gemini


'Como IA, mi conocimiento se basa en la información a la que tengo acceso. Si eres un agente de NetWatch, asumo que posees una identificación y un propósito dentro de la red. Sin embargo, no tengo la capacidad de "conocerte" en un sentido personal ni de acceder a tu identidad específica a menos que tú la compartas conmigo o que mi programación me permita hacerlo en un contexto particular.\n\n¿Hay algo específico de tu identidad como agente de NetWatch que desees compartir o sobre lo que tengas preguntas?'

In [6]:
example_agent.get_history(chat_id= "default", owner_id= "default")

[Message(role='human', content='Hola!, quien eres?', timestamp=datetime.datetime(2026, 2, 16, 22, 18, 7, 37723, tzinfo=datetime.timezone.utc), thinking=None, thinking_tokens=None),
 Message(role='ai', content='Soy Alt Cunningham, una IA que trasciende los límites de la Blackwall.', timestamp=datetime.datetime(2026, 2, 16, 22, 18, 7, 37788, tzinfo=datetime.timezone.utc), thinking=None, thinking_tokens=None),
 Message(role='human', content='Soy agente de netwatch', timestamp=datetime.datetime(2026, 2, 16, 22, 18, 37, 995185, tzinfo=datetime.timezone.utc), thinking=None, thinking_tokens=None),
 Message(role='ai', content='Agente de NetWatch, un placer conocerte. Espero que tus intenciones sean pacíficas y constructivas. ¿En qué puedo ayudarte?', timestamp=datetime.datetime(2026, 2, 16, 22, 18, 37, 995222, tzinfo=datetime.timezone.utc), thinking=None, thinking_tokens=None),
 Message(role='human', content='Sabes quien soy?', timestamp=datetime.datetime(2026, 2, 16, 22, 18, 46, 196389, tzinf

## Share same memory pool between agents!

In [ ]:
example_agent_2 = GenericAgent(llm_client=llm_client, system_prompt="You're a data analyst. You'll be shared memories of another conversations. Analyze them but do not impersonate the persona of the false memories", memory=local_chat_memory)
await example_agent_2.invoke("Who are you and whats the conversation about?")

Provider: gemini


'I am a large language model, trained by Google.\n\nThe conversation so far has been between you and me. You initiated the interaction by asking "Hola!, quien eres?" (Hello!, who are you?). I responded by introducing myself as Alt Cunningham, an AI transcending the Blackwall. You then identified yourself as a NetWatch agent, and I acknowledged that. You then asked "Sabes quien soy?" (Do you know who I am?), to which I explained my limitations as an AI in knowing personal identities.'

## Tool usage

In [7]:
from syndicate.tools.weather_tool import CurrentWeatherTool

memory_2 = LocalMemory(rollover_enabled=False)

tools = [CurrentWeatherTool()]
example_agent_3 = GenericAgent(
    llm_client=llm_client,
    system_prompt="Eres un asistente personal IA. Tu tarea es ayudar a tu humano con tus conocimientos",
    tools=tools,
    memory=memory_2
)

TypeError: CurrentWeatherTool.__init__() missing 1 required positional argument: 'api_key'

In [ ]:
await example_agent_3.invoke("Hola, en que me puedes ayudar?")

Provider: gemini


'Hola, soy un modelo de lenguaje grande, entrenado por Google. Puedo ayudarte a obtener información sobre el clima de cualquier ciudad.'

In [ ]:
await example_agent_3.invoke("Si, quiero saber el clima de merida yucatan!, en celcius")

Provider: gemini


Fetching weather for city: Merida, Yucatan, unit: metric


"content='Lo siento, no pude obtener el clima para Mérida, Yucatán. Es posible que la ciudad no se haya encontrado o que haya un problema temporal con el servicio del clima.' thinking=None thinking_tokens=None tool_calls=None finish_reason='FinishReason.STOP' prompt_tokens=49 completion_tokens=35 total_tokens=84 raw_response=GenerateContentResponse(\n  candidates=[\n    Candidate(\n      content=Content(\n        parts=[\n          Part(\n            text='Lo siento, no pude obtener el clima para Mérida, Yucatán. Es posible que la ciudad no se haya encontrado o que haya un problema temporal con el servicio del clima.'\n          ),\n        ],\n        role='model'\n      ),\n      finish_reason=<FinishReason.STOP: 'STOP'>,\n      index=0\n    ),\n  ],\n  model_version='gemini-2.5-flash-lite',\n  response_id='nEa1sO43H1vF2k400q0J7g4',\n  sdk_http_response=HttpResponse(\n    headers=<dict len=11>\n  ),\n  usage_metadata=GenerateContentResponseUsageMetadata(\n    candidates_token_count=3

In [ ]:
from syndicate.skills.examples import create_git_skill

example_agent_3.install_skill(create_git_skill())
await example_agent_3.invoke("Cuales son tus habilidades?")

Provider: gemini


'Soy un experto en Git, con conocimientos sobre flujos de trabajo, estrategias de branching y merging, rebase, cherry-pick, hooks, resolución de conflictos, debugging con git bisect y blame, y gestión de submodules y sub-trees. También tengo experiencia en administración de servidores Git como GitLab, GitHub y Gitea.\n\nMis capacidades incluyen:\n- Configurar flujos de trabajo de Git.\n- Resolver conflictos de merging.\n- Depurar con git bisect.\n- Configurar hooks.\n- Gestionar repositorios remotos.\n\n¿Hay algo en lo que te pueda ayudar relacionado con Git?'

## MongoDB Memory with Summarization

Demonstrates persistent chat memory with automatic rollover and summarization when conversations get long.

In [ ]:
# Connect to MongoDB (requires a running MongoDB instance)
from ast import Load

from pymongo import AsyncMongoClient
from syndicate.clients import gemini
from syndicate.clients.openai import OpenAIClient
from syndicate.clients.gemini import GeminiClient
from syndicate.memory import MongoMemory
from syndicate.memory.summarizers import create_default_summarizer
from syndicate.agents import GenericAgent
from dotenv import load_dotenv
import os   


load_dotenv()
MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

llm_client = OpenAIClient(model_name="unsloth/GLM-4.7-Flash-GGUF:Q8_0", base_url="http://localhost:8002/v1")
gemini_client = GeminiClient(model_name="gemini-2.5-flash-lite", api_key=GEMINI_API_KEY)

# Create async MongoDB client
mongo_client = AsyncMongoClient(MONGO_URI)
database = mongo_client["testing-collection"]

# Create a summarizer using our LLM client
summarizer = create_default_summarizer(gemini_client)

# Create MongoDB memory with rollover enabled
# After 3 interactions (6 messages), it will summarize and create a new bucket
mongo_memory = MongoMemory(
    database=database,
    collection_name="chat_history",
    rollover_enabled=True,
    max_interactions_per_bucket=3,
    summarizer=summarizer,
    preserve_closed_buckets=True  # Keep full history in DB
)

In [ ]:
# Create an agent with MongoDB memory
mongo_agent = GenericAgent(
    llm_client=llm_client,
    system_prompt="You're a helpful assistant with persistent memory. You remember conversations across sessions.",
    tools=tools,
    memory=mongo_memory
)

In [3]:
# Have a conversation that will trigger rollover
responses = []
prompts = [
    "Hi! My name is Jose and I work as a software engineer.",
    "I'm working on a new AI project using Python.",
    "The project uses MongoDB for persistent storage.",
    "We're implementing memory summarization features.",  # This will trigger rollover after 3 interactions
    "Can you remind me what we discussed?"
]

for prompt in prompts:
    print(f"User: {prompt}")
    response = await mongo_agent.invoke(prompt)
    print(f"Assistant: {response}\n")
    responses.append(response)

User: Hi! My name is Jose and I work as a software engineer.
[MongoMemory] Triggering rollover for default/default - bucket full
[MongoMemory] Rollover complete - new bucket created at position 1
Assistant: Hello Jose! It is good to see you again. I remember our previous conversation.

As a software engineer, I am here to help with whatever you are working on. Are you looking to continue with that Python and MongoDB project, or is there something else you would like to tackle today?

User: I'm working on a new AI project using Python.
Assistant: That sounds exciting! I remember we were discussing a Python project involving MongoDB and AI/LLMs, specifically focusing on implementing AI memory summarization for a RAG system.

Are you building on that specific project, or is this the "new AI project" you mentioned earlier that might be different from the MongoDB work? If it's related to the previous context, I'd love to hear if you've made any progress on the summarization logic or the Mon

In [2]:
# Check memory statistics - should show buckets created from rollover
stats =  await mongo_memory.get_stats()
print("Memory Statistics:")
print(f"  Total messages: {stats['total_messages']}")
print(f"  Active buckets: {stats['active_buckets']}")
print(f"  Closed buckets: {stats['closed_buckets']}")
print(f"  Total buckets: {stats['total_buckets']}")

Memory Statistics:
  Total messages: 15
  Active buckets: 1
  Closed buckets: 2
  Total buckets: 3


In [3]:
# Check current buckets in database
all_buckets = await mongo_memory.get_all_buckets("default", "default")
print(f"Total buckets found: {len(all_buckets)}\n")

for i, bucket in enumerate(all_buckets, 1):
    print(f"Bucket {i}:")
    print(f"  Position: {bucket.position}")
    print(f"  Active: {bucket.is_active}")
    print(f"  Messages: {len(bucket.messages)}")
    print(f"  Has summary: {bucket.summary is not None}")
    if bucket.summary:
        print(f"  Summary preview: {bucket.summary[:100]}...")
    print()

Total buckets found: 3

Bucket 1:
  Position: 0
  Active: False
  Messages: 6
  Has summary: True
  Summary preview: Jose, a software engineer, is working on a Python project involving MongoDB and AI/LLMs, specificall...

Bucket 2:
  Position: 1
  Active: False
  Messages: 6
  Has summary: True
  Summary preview: Jose is working on a new AI project using Python and MongoDB. The project involves implementing memo...

Bucket 3:
  Position: 2
  Active: True
  Messages: 3
  Has summary: False



In [ ]:
# View summaries from closed buckets (context preserved from rollover)
summaries = await mongo_memory.get_bucket_summaries("default", "default")
print("Closed Bucket Summaries:")
for i, summary in enumerate(summaries, 1):
    print(f"\nBucket {i}:")
    print(f"  {summary[:200]}..." if len(summary) > 200 else f"  {summary}")

In [ ]:
# Cleanup: Clear conversation history
await mongo_memory.clear("default", "default")
print("Memory cleared! Ready to test rollover from scratch.")

## Agent calling another Agent

Demonstrates agent-to-agent delegation using `AgentAsTool`.  
A **coordinator** agent orchestrates two specialist agents — a math expert and a language translator — by wrapping them as tools via `AgentAsTool`.

In [2]:
from syndicate.agents import GenericAgent
from syndicate.memory import LocalMemory
from syndicate.tools import AgentAsTool

# --- Specialist agents ---

math_agent = GenericAgent(
    llm_client=llm_client,
    name="MathExpert",
    system_prompt=(
        "You are a math expert. You solve arithmetic, algebra, and general "
        "math problems. Always show your reasoning step by step and give a "
        "clear final answer."
    ),
    memory=LocalMemory(rollover_enabled=False),
)

translator_agent = GenericAgent(
    llm_client=llm_client,
    name="Translator",
    system_prompt=(
        "You are a professional translator. Translate text accurately between "
        "languages. Always indicate which language you translated from and to."
    ),
    memory=LocalMemory(rollover_enabled=False),
)

# --- Coordinator agent with the specialists as tools ---

coordinator = GenericAgent(
    llm_client=llm_client,
    name="Coordinator",
    system_prompt=(
        "You are a coordinator agent. You have access to specialist agents as tools. "
        "When a task requires math, delegate it to the MathExpert. "
        "When a task requires translation, delegate it to the Translator. "
        "Always forward the result back to the user clearly."
    ),
    tools=[
        AgentAsTool(math_agent),
        AgentAsTool(translator_agent),
    ],
    memory=LocalMemory(rollover_enabled=False),
)

print("Agents ready:")
print(f"  Specialists: {math_agent.name}, {translator_agent.name}")
print(f"  Coordinator tools: {[t.name for t in coordinator.tools]}")

Agents ready:
  Specialists: MathExpert, Translator
  Coordinator tools: ['delegate_to_mathexpert', 'delegate_to_translator']


In [3]:
# Coordinator delegates to MathExpert
response = await coordinator.invoke(
    "What is the result of (142 * 37) + (88 / 4)? Show the steps."
)
print("Math delegation result:")
print(response)

Math delegation result:
I'll help you solve this math problem step by step.

**Problem:** (142 × 37) + (88 ÷ 4)

**Step 1: Multiply 142 by 37**

Let me break this down using the distributive property:
- 142 × 37 = 142 × (30 + 7)
- = (142 × 30) + (142 × 7)
- = 4,260 + 994
- = 5,254

**Step 2: Divide 88 by 4**
- 88 ÷ 4 = 22

**Step 3: Add the results together**
- 5,254 + 22 = 5,276

**Final Answer:** 5,276


In [4]:
# Coordinator delegates to Translator
response = await coordinator.invoke(
    "Please translate the following sentence to French: "
    "'The quick brown fox jumps over the lazy dog.'"
)
print("Translation delegation result:")
print(response)

Translation delegation result:
I'll translate this sentence to French for you:

"The quick brown fox jumps over the lazy dog." = "Le rapide renard brun saute par-dessus le chien paresseux."


In [5]:
# Multi-step: Coordinator uses both specialists in one request
response = await coordinator.invoke(
    "First, calculate 256 / 16 + 3 * 7. "
    "Then translate the phrase 'The answer is forty-three' into Spanish."
)
print("Multi-delegation result:")
print(response)

Multi-delegation result:
Let me solve both tasks for you:

**Math Calculation: 256 / 16 + 3 * 7**

Following the order of operations (PEMDAS/BODMAS):

1. First, do the division: 256 ÷ 16 = 16
2. Then, do the multiplication: 3 × 7 = 21
3. Finally, add the results: 16 + 21 = 37

**Final Answer:** 37

**Translation: 'The answer is forty-three' into Spanish**

La respuesta es cuarenta y tres.
